# 1000 Genomes Phase 3

Precomputed ADMIXTURE results (K=2…8, 2,426 individuals) from
[pong](https://github.com/ramachandran-lab/pong)'s example. The true grouping
(5 continental regions) is known, so we can check the result.

1. **stripesankey**: clusters across K
2. **metasankey**: the same view, coloured by metadata
3. **affinity heatmap**: what the clusters are built from

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path.cwd()
POP = HERE.parent if HERE.name == 'notebooks' else HERE
sys.path.insert(0, str(POP.parent.parent / 'src'))   # the local stripesankey

import stripesankey as ss

BASE = POP / 'stripesankey_output'
BASE

## Load data

In [ ]:
W = {int(p.stem.split('_')[1]): pd.read_csv(p, index_col=0)
     for p in sorted((BASE / 'results' / 'W').glob('W_*.csv'))}
K_LIST = sorted(W)

metadata = pd.read_csv(POP / 'data' / 'metadata.csv', index_col=0)
with open(BASE / 'results' / 'consensus.json') as f:
    consensus = json.load(f)
clusters = {s: int(c) for s, c in consensus['sample_to_cluster'].items()}

print(f"W: K={K_LIST}, {W[K_LIST[0]].shape[0]} individuals")
print(f"consensus across K={consensus['k_pooled']}: {consensus['sizes']}")
print(f"metadata: {metadata['population'].nunique()} populations in "
      f"{metadata['region'].nunique()} regions")
metadata.head()

### Does the consensus match the regions?

The consensus never sees `region`.

In [ ]:
from sklearn.metrics import adjusted_rand_score

meta_plus = metadata.join(pd.Series(clusters, name='consensus'))
print('ARI(consensus, region) = '
      f"{adjusted_rand_score(meta_plus['consensus'], meta_plus['region']):.4f}")
pd.crosstab(meta_plus['consensus'], meta_plus['region'])

AFR, EAS and SAS are recovered; EUR and AMR merge (they share ancestry).

## 1 — stripesankey

One column per K, coloured by consensus cluster.
Click a ribbon or a band to trace those individuals across K.

In [ ]:
HOVER = ['region', 'population']

# The palette and the legend tree are not passed: `stripesankey` builds both
# from the same chance-corrected Q it clustered, so the left-column legend
# agrees with the ribbons by construction. Override that pair only together.
stripe = ss.stripesankey(W, clusters=clusters, k_list=K_LIST,
                         metadata_df=metadata, hover_field=HOVER,
                         figsize=(3, 5.2))
stripe

Click something above, then run the next cell.

In [ ]:
sel = stripe.selected_samples or []
if sel:
    sub = meta_plus.reindex([s for s in sel if s in meta_plus.index])
    print(f'{len(sel)} individuals selected')
    display(pd.concat([sub[c].value_counts().rename('n').to_frame().assign(field=c)
                       for c in ('region', 'population', 'consensus')]))
else:
    print('nothing selected yet — click a ribbon or a node band above')

## 2 — metasankey

Same layout, coloured by a metadata field instead.

In [ ]:
FIELDS = ['region', 'population']   # the first one colours nodes and flows

meta_stripe = ss.metasankey(W, clusters=clusters, k_list=K_LIST,
                            metadata_df=metadata, hover_field=FIELDS,
                            figsize=(3, 5.2))
meta_stripe

## 3 — Affinity heatmap

How often two individuals share components across K (positive = more than chance).
The blocks are the consensus clusters.

In [ ]:
Q = ss.consensus_affinity({k: w.to_numpy() for k, w in W.items()})
Q = pd.DataFrame(Q, index=W[2].index, columns=W[2].index)
print(f'Q: {Q.shape}, {Q.to_numpy().nbytes / 1e6:.0f} MB in memory')

fig, order, labels = ss.affinity_heatmap(
    Q,
    above=meta_plus[['region']],
    annotations=['region'],
    figsize=(6.4, 7.2),
    title='1000 Genomes Phase 3 — consensus affinity across K=2…8',
)
fig

### Regenerating the data

```bash
python ../scripts/build_from_pong.py <path to 1kg_p3>
```